Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Demo 12 - Machine Learning

"All models are wrong, but some are useful." (George Box, 1976)

How would you prefer your model to be wrong so that it is most useful?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn import tree

import demo12

%load_ext autoreload
%autoreload 2

# Data Loading and Wrangling

In [ ]:
# Load filings
df = pd.read_csv('District_Court_of_Maryland_Eviction_Case_Data_MG_PG.csv', low_memory=False)

## Construct tidy table for modeling units

In [ ]:
# Make eviction indicator
df['evicted'] = df['Eviction Year'].notnull().astype(int)

In [ ]:
# Convert dates stored as strings to datetime objects 
df['Event Date'] = pd.to_datetime(df['Event Date'], format='mixed')
df['Evicted Date'] = pd.to_datetime(df['Evicted Date'], format='mixed')

# Convert zip codes to strings without decimals
df['Tenant ZIP Code'] = df['Tenant ZIP Code'].astype('Int64').astype('string')

In [ ]:
# Retrieve first, min, or max values for each case
agg_funcs = {
    'Event Date': 'min',
    'County': 'first',
    'Tenant City': 'first',
    'Tenant State': 'first',
    'Tenant ZIP Code': 'first',
    'Case Type': 'first',
    'evicted': 'max',
}
df = df.groupby('Case Number').agg(agg_funcs)

## Data Cleaning

In [ ]:
df['Case Type'].value_counts()

In [ ]:
# Limit to tenants with Maryland addresses
df = df[df['Tenant State'] == 'MD']

In [ ]:
# Recode 'Case Type' value that's spelled wrong
df['Case Type'] = df['Case Type'].replace('Breach Of Lease', 'Breach of Lease')

## Construct new features

In [ ]:
# Calculate a feature for month of first filing
df['event_month'] = df['Event Date'].dt.month

# Make month dummies
df = pd.concat([df, pd.get_dummies(df['event_month'], prefix='event_month').astype(int)], axis=1)

# Calculate a feature for the number of days before the end of the month on which an event was first filed
df['event_days_before_end_of_month'] = df['Event Date'].dt.daysinmonth - df['Event Date'].dt.day

# Case case type dummies
df = pd.concat([df, pd.get_dummies(df['Case Type'], prefix='case_type').astype(int)], axis=1)

# Make county dummies
df = pd.concat([df, pd.get_dummies(df['County'], prefix='county').astype(int)], axis=1)

# Make zip dummies
df = pd.concat([df, pd.get_dummies(df['Tenant ZIP Code'], prefix='zip').astype(int)], axis=1)

In [ ]:
df.columns.tolist()

# Model Specification

## Define Features and Outcome Variable

Want enough features with well-theorized relationships to the outcome to achieve a strong and generalizable model fit.

- Want to avoid underfitting
- Want to avoid overfitting

<img alt="Underfitting and Overfitting Example" src='https://miro.medium.com/v2/resize:fit:720/format:webp/0*vayIXMjSp3ezj4G6.png'>

[Source: https://medium.com/@satyam3196/everything-you-need-to-know-about-model-fitting-in-machine-learning-4f93dccc6bf1](https://medium.com/@satyam3196/everything-you-need-to-know-about-model-fitting-in-machine-learning-4f93dccc6bf1)

In [ ]:
# Split dataset in features and target variable
features = [ 
    # Date
    'event_month',
    # 'event_month_1',
    # 'event_month_2',
    # 'event_month_3',
    # 'event_month_4',
    # 'event_month_5',
    # 'event_month_6',
    # 'event_month_7',
    # 'event_month_8',
    # 'event_month_9',
    # 'event_month_10',
    # 'event_month_11',
    # 'event_month_12',
    'event_days_before_end_of_month',
    
    # Case type
    # 'case_type_Breach of Lease',
    'case_type_Failure to Pay Rent',
    'case_type_Tenant Holding Over',
    'case_type_Wrongful Detainer',
    
    # County
    # 'county_Montgomery',
    "county_Prince George's",
    
    # Zip Code
    # 'zip_20375',
    # 'zip_20447',
    # 'zip_20474',
    # 'zip_20555',
    # 'zip_20580',
    # 'zip_20601',
    # 'zip_20603',
    # 'zip_20607',
    # 'zip_20608',
    # 'zip_20613',
    # 'zip_20623',
    # 'zip_20679',
    # 'zip_20701',
    # 'zip_20702',
    # 'zip_20703',
    # 'zip_20705',
    # 'zip_20706',
    # 'zip_20707',
    # 'zip_20708',
    # 'zip_20710',
    # 'zip_20711',
    # 'zip_20712',
    # 'zip_20713',
    # 'zip_20715',
    # 'zip_20716',
    # 'zip_20720',
    # 'zip_20721',
    # 'zip_20722',
    # 'zip_20723',
    # 'zip_20724',
    # 'zip_20725',
    # 'zip_20735',
    # 'zip_20737',
    # 'zip_20738',
    # 'zip_20740',
    # 'zip_20742',
    # 'zip_20743',
    # 'zip_20744',
    # 'zip_20745',
    # 'zip_20746',
    # 'zip_20747',
    # 'zip_20748',
    # 'zip_20749',
    # 'zip_20757',
    # 'zip_20767',
    # 'zip_20769',
    # 'zip_20770',
    # 'zip_20772',
    # 'zip_20773',
    # 'zip_20774',
    # 'zip_20779',
    # 'zip_20781',
    # 'zip_20782',
    # 'zip_20783',
    # 'zip_20784',
    # 'zip_20785',
    # 'zip_20786',
    # 'zip_20788',
    # 'zip_20796',
    # 'zip_20805',
    # 'zip_20806',
    # 'zip_20810',
    # 'zip_20814',
    # 'zip_20815',
    # 'zip_20816',
    # 'zip_20817',
    # 'zip_20818',
    # 'zip_20831',
    # 'zip_20832',
    # 'zip_20833',
    # 'zip_20837',
    # 'zip_20841',
    # 'zip_20842',
    # 'zip_20847',
    # 'zip_20849',
    # 'zip_20850',
    # 'zip_20851',
    # 'zip_20852',
    # 'zip_20853',
    # 'zip_20854',
    # 'zip_20855',
    # 'zip_20857',
    # 'zip_20860',
    # 'zip_20861',
    # 'zip_20862',
    # 'zip_20866',
    # 'zip_20868',
    # 'zip_20871',
    # 'zip_20872',
    # 'zip_20874',
    # 'zip_20875',
    # 'zip_20876',
    # 'zip_20877',
    # 'zip_20878',
    # 'zip_20879',
    # 'zip_20882',
    # 'zip_20884',
    # 'zip_20886',
    # 'zip_20895',
    # 'zip_20898',
    # 'zip_20901',
    # 'zip_20902',
    # 'zip_20903',
    # 'zip_20904',
    # 'zip_20905',
    # 'zip_20906',
    # 'zip_20907',
    # 'zip_20908',
    # 'zip_20910',
    # 'zip_20912',
    # 'zip_20914',
    # 'zip_20918',
    # 'zip_20985',
    # 'zip_20988',
    # 'zip_21037',
    # 'zip_21046',
    # 'zip_21069',
    # 'zip_21078',
    # 'zip_21202',
    # 'zip_21212',
    # 'zip_21217',
    # 'zip_21218',
    # 'zip_21229',
    # 'zip_21239',
    # 'zip_21401',
    # 'zip_21404',
    # 'zip_21601',
    # 'zip_21703',
    # 'zip_21710',
    # 'zip_21746',
    # 'zip_21771',
    # 'zip_21853',
    # 'zip_22003',
    # 'zip_22191',
    # 'zip_22769',
    # 'zip_27048',
    # 'zip_28014',
    # 'zip_29094',
    # 'zip_29747',
    # 'zip_29748',
]

X = df[features] # Features
y = df.evicted # Outcome variable

## Split Data for Training and Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
len(X_train)

In [ ]:
len(X_test)

# Logistic Regression

## Train

In [ ]:
# Estimate with statsmodels (best for statistical inference/interpretation)
# Uses Maximum Likelihood Estimation (MLE) by default  
sm.Logit(y_train, X_train).fit().summary()

In [ ]:
# Estimate with sklearn (best for predictive modeling and consistency with more advanced ML methods)
# Uses L-BFGS (Limited-memory Broyden–Fletcher–Goldfarb–Shanno) solver by default

# Initialize the model object
logreg = LogisticRegression(random_state=1, max_iter=100)

# Estimate/fit the model
logreg.fit(X_train, y_train)

In [ ]:
# Note that the coefficients from by L-BFGS estimation are similar but not identical to those from SME
logreg.coef_

## Test

In [ ]:
# By default, the model predicts an outcome of level 1 (eviction)
# if the probability is greater than 0.5
y_pred = logreg.predict(X_test)
y_pred

In [ ]:
# Alternatively, we could generalize prediction with a custom probability threshold  
def predict_with_threshold(model, X_test, threshold=0.5):
    y_pred_proba = model.predict_proba(X_test)[:,1]
    return (y_pred_proba > threshold).astype(int)

y_pred = predict_with_threshold(logreg, X_test, threshold=0.5)
y_pred

In [ ]:
demo12.confusion_matrix(y_test, y_pred)

# Assessing Model "Fit"

This is always going to be a balance between more and less sensitivity to the outcome you want to predict.

Do you want more or fewer false alarms? How much do you want to "cry wolf"?

### Accuracy
Proportion of classifications that are correct.

$$
Accuracy = \frac{Correct Classifications}{Total Classifications} = \frac{True Positives + True Negatives}{True Positives + True Negatives + False Positives + False Negatives}
$$

### Precision/Specificity
Proportion of positive predictions that are correct.

You want precision to be high if it's bad to have false positives (e.g., if unnecessary identification and treatment is costly).

$$
Precision = \frac{True Positives}{True Positives + False Positives}
$$

### Recall/Sensitivity/True Positive Rate
Proportion of actual positives that are predicted correctly.

You want recall to be high if it's bad to have false negatives (e.g., if you would rather identify and treat even if it's not strictly necessary).

$$
Recall = \frac{True Positives}{True Positives + False Negatives}
$$

In [ ]:
threshold = 0.2

y_pred = predict_with_threshold(logreg, X_test, threshold)
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)

print(f'Accuracy:   {accuracy:.2f}  (% of predictions that are correct)')
print(f'Precision:  {precision:.2f}  (% of predicted evictions that are correct)')
print(f'Recall:     {recall:.2f}  (% of actual evictions that are predicted correctly)')


In [ ]:
demo12.plot_fit_stats(logreg, X_test, y_test)

### Receiver Operating Characteristic (ROC) Curve

Built by iterating classification thresholds between 0 and 1, then plotting sensitivity against specificity for the resulting outcomes.

Shows the performance of the model at all possible classification thresholds.

The closer the curve is to a right angle, the better the model.

Area Under the Curve (AUC) summarizes performance; better models approach 1.0.

In [ ]:
demo12.roc_plot(logreg, y_test, X_test)

# Random Forest (Tree-Based Classifer)

Based on decision trees:

<img alt="Gini impurity visualization" src='https://media.geeksforgeeks.org/wp-content/uploads/20250408153824016146/predicting_whether_a_customer_will_buy_a_product.webp' width="600">

[Source: https://www.geeksforgeeks.org/machine-learning/decision-tree-introduction-example/](https://www.geeksforgeeks.org/machine-learning/decision-tree-introduction-example/)

Trees are constructed by minimizing Gini Impurity at each node:

<img alt="Gini impurity visualization" src='https://www.pythonalchemist.com/_next/image?url=%2Fimages%2Fcovers%2Fgini-impurity-visualizer.webp&w=1920&q=75' width="600">

[Source: https://williamkoehrsen.medium.com/random-forest-simple-explanation-377895a60d2d](https://williamkoehrsen.medium.com/random-forest-simple-explanation-377895a60d2d)

A random forest is a collection of (many, many) decision trees:

<img alt="Random forest decision tree example" src='https://miro.medium.com/v2/resize:fit:1184/format:webp/1*i0o8mjFfCn-uD79-F1Cqkw.png'>

[Source: https://www.pythonalchemist.com/blog/gini-impurity-visualizer](https://www.pythonalchemist.com/blog/gini-impurity-visualizer)

If you want to dive deeper into decision trees and to train them, [this is an excellent YouTube primer.](https://www.youtube.com/watch?v=B6a64wdD7Zs)
## Train

In [ ]:
# Initialize the random forest classifer object
rf = RandomForestClassifier(n_estimators = 50, random_state = 1, n_jobs=-1, max_depth=4) 

# Fit the model based on training data
rf.fit(X_train, y_train)

# Visualize the first estimator/tree
e = rf.estimators_[0]

fig, ax = plt.subplots(figsize=(40,20))  
_ = tree.plot_tree(e, feature_names = X_train.columns, fontsize=15, ax = ax)

In [ ]:
## Now let's allow the depth of decision trees to be unlimited
## (Nodes are expanded until each leaf contains one observation)

# Initialize the random forest classifer object
rf = RandomForestClassifier(n_estimators = 100, random_state = 1, max_depth=None)

# Fit the model
rf.fit(X_train, y_train)

## Test

In [ ]:
y_pred = rf.predict(X_test)
y_pred

In [ ]:
demo12.confusion_matrix(y_test, y_pred)

In [ ]:
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)

print(f'Accuracy:   {accuracy:.2f}  (% of predictions that are correct)')
print(f'Precision:  {precision:.2f}  (% of predicted evictions that are correct)')
print(f'Recall:     {recall:.2f}  (% of actual evictions that are predicted correctly)')

In [ ]:
demo12.roc_plot(rf, y_test, X_test)

In [ ]:
# Investigate feature importance with mean decrease in impurity (MDI)

importances = pd.Series(rf.feature_importances_, index=X_test.columns)
std = np.std([tree.feature_importances_ for tree in rf.estimators_], axis=0)
fig, ax = plt.subplots()
importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()